# CO1 JAVA – Campus Smart-Parking System
## Assignment 1 (Tasks 1–12)

**Student:** Archana V  
**Register Number:** 192521420  
**Faculty:** Ravi Pujitha  
**Academic Year:** 2026–2027  
**Course Outcome:** CO1 – Java fundamentals and object-oriented programming  
**Assignment title:** Design, Implement and Validate an Object-Oriented Model for a Campus Smart-Parking System

This notebook follows the uploaded assignment PDF task-by-task. It contains analysis, pseudocode, Java examples, test evidence, UML, alternatives and Big-O comparisons.

> Run cells in order. The Java execution cells use OpenJDK and write source files to the Colab working directory. Java examples are designed for Java 17 or newer.

## 0. Set up Java in Google Colab

Colab normally provides Python. This setup installs OpenJDK so the Java source cells can compile and run. If Java is already installed, this command still verifies the environment.

In [1]:
!apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless
!java -version
!javac -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.21.0.3+1-1~24.04.1_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.21.0.3+1-1~24.04.1) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.21.0.3+1-1~24.04.1_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.21.0.3+1-1~24.04.1) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.21.0.3+1-1~24.04.1) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.044s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't

# Task 1 – Understand the Problem and Identify the Objects (4 marks)
**Bloom’s Level: K4 – Analyse**

### 1.1 Inputs and outputs
**Inputs:** registration number, vehicle category, entry time, exit time/duration, floor/slot configuration, pass type, and charging energy (for EVs).

**Outputs:** allocated floor and slot, unique ticket number, entry/exit records, fee, occupancy status, error messages for invalid/full/duplicate requests, and collection totals.

### 1.2 Nouns → candidate classes and verbs → methods
| Candidate class | Responsibility / reason it is needed | Candidate methods |
|---|---|---|
| `Vehicle` | Shared registration identity and common fee behavior | `getRegNo()`, `calculateFee()` |
| `TwoWheeler`, `Car`, `ElectricCar` | Vehicle-specific pricing/capability | `ratePerHour()`, `chargingFee()` |
| `ParkingSlot` | Holds slot identity and current occupant; prevents double occupancy | `isFree()`, `park()`, `release()` |
| `ParkingFloor` | Groups slots and searches a floor | `findFreeSlot()` |
| `ParkingFacility` | Owns floors, allocates/releases slots | `allocate()`, `release()` |
| `ParkingTicket` | Ticket ID, registration, location and entry time | `close()`, getters |
| `ParkingPass` / `PassHolder` | Represents eligibility for free/discount parking | `isValid()` |
| `ParkingService` | Coordinates entry, exit, tickets and fee computation | `parkVehicle()`, `exitVehicle()` |

The nouns identify stateful entities; the verbs identify operations that enforce rules. A class is justified when it has a distinct responsibility or behavior, rather than placing all logic in one oversized class.

### 1.3 Business rules and assumptions
- A slot can contain at most one vehicle; only a free slot can be allocated.
- Ticket numbers are unique and never reused during the system lifetime.
- Registration numbers are unique among currently parked vehicles.
- Allocation returns the first available suitable slot; if none exists, parking fails cleanly.
- Exit validates the active ticket and slot before releasing it.
- Started hours are billed as whole hours in Task 8; the detailed Task 11 rates are explicitly defined in its code.
- Staff/students with valid monthly passes park free; visitors pay.
- EV charging is an additional fee, separate from parking fee.
- Times are assumed to be local campus time; duration must be non-negative.
- For this model, floor and slot indexes are zero-based internally; display labels may be one-based.

### 1.4 Alternative: objects vs parallel arrays
**Object model:** `Vehicle` stores registration and category together. Searching active vehicles is linear if a list is used: O(n). Each object stores fields plus object/reference overhead, but adding an attribute means changing a class and constructors/getters.

**Parallel arrays:** `regNo[]`, `type[]`, `entryTime[]` keep related fields at the same index. Search is O(n); adding a new attribute requires another array and keeping every index synchronized. Per vehicle, storage is O(1) fields but several arrays and synchronization risks. Object design is easier to maintain and less error-prone as categories grow.

# Task 2 – Choose the Data Representation (3 marks)
**Bloom’s Level: K4 – Analyse**

| Field | Type | Reason |
|---|---|---|
| Floor/slot index | `int` | Whole-number positions |
| Registration number | `String` | Alphanumeric vehicle identifier |
| Entry/exit time | `LocalDateTime` or epoch milliseconds (`long`) | Accurate time representation |
| Duration | `long` minutes | Whole minute billing input |
| Fee | `double` for simple example; `BigDecimal` recommended for money | Avoid binary floating-point money errors in production |
| Occupant | `Vehicle` reference or `null` | Object identity/empty slot |

### Array representation
A jagged array `String[][] slots` is appropriate because each floor can have a different slot count: `slots[floor][slot]`. It is simple and maps directly to floors. A one-dimensional array is also possible if every floor has equal capacity; the index is `floor * slotsPerFloor + slot`. An `ArrayList<String[]>` is flexible for variable floor sizes but adds collection overhead.

### Type conversion / precision risk
Incorrect integer division: `90 / 60` is `1`, so using it as a duration in hours would undercharge 90 minutes. Casting after division (`(double)(90 / 60)`) still produces 1.0. Correct: `Math.ceil(90 / 60.0)` or integer ceiling formula `(minutes + 59) / 60`.

### Input
Use `Scanner` for interactive input; command-line arguments are convenient for scripted test cases. Validate registration, numeric values, and duration before changing facility state.

### Alternative & complexity
- Jagged array: locating a known floor/slot is O(1); scanning for a free slot is O(F×S) worst-case. Space O(F×S). Supports different slot counts per floor.
- Flat array: O(1) access by computed index, but different floor capacities need offsets/prefix sums or a mapping. Space O(F×S).
- `ArrayList<String[]>`: per-floor access O(1), scanning O(F×S); variable floor lengths naturally supported, with collection overhead.

# Task 3 – Design the Control Logic (4 marks)
**Bloom’s Level: K4 – Analyse**

### 3.1 Pseudocode: allocate a slot
```text
ALGORITHM Allocate(vehicleNo)
    FOR each floor f from 0 to floorCount - 1
        FOR each slot s from 0 to slots[f].length - 1
            IF slots[f][s] is NULL THEN
                slots[f][s] ← vehicleNo
                RETURN (f, s)
            END IF
        END FOR
    END FOR
    RETURN NULL  // no free slot
END ALGORITHM
```

### 3.2 Pseudocode: fee calculation
```text
ALGORITHM CalculateFee(minutes, hourlyRate)
    IF minutes <= 0 THEN RETURN 0
    hours ← (minutes + 59) DIV 60  // ceiling using integer arithmetic
    fee ← hours × hourlyRate
    IF fee > dailyMaximum THEN fee ← dailyMaximum
    RETURN fee
END ALGORITHM
```

Loops stop on the first free slot using `return`. `continue` may skip occupied slots; `break` can stop an inner scan if a slot is found, while `return` exits the method directly. If no slot is free, return `null` and do not overwrite any vehicle. The sample code below preserves the assignment’s supplied `main()` method.

In [2]:
%%writefile ParkingFacility.java
public class ParkingFacility {
    // slots[floor][slot] holds a vehicle number, or null if the slot is free
    private final String[][] slots;

    public ParkingFacility(int[] slotsPerFloor) {
        slots = new String[slotsPerFloor.length][];
        for (int f = 0; f < slotsPerFloor.length; f++) {
            slots[f] = new String[slotsPerFloor[f]];
        }
    }

    /** Returns {floor, slot} of the first free slot, or null if it is full. */
    public int[] allocate(String vehicleNo) {
        if (vehicleNo == null || vehicleNo.isBlank()) return null;
        for (int f = 0; f < slots.length; f++) {
            for (int s = 0; s < slots[f].length; s++) {
                if (slots[f][s] == null) {
                    slots[f][s] = vehicleNo;
                    return new int[]{f, s};
                }
            }
        }
        return null;
    }

    /** Frees a slot. Returns false if the position is invalid or already free. */
    public boolean release(int floor, int slot) {
        if (floor < 0 || floor >= slots.length) return false;
        if (slot < 0 || slot >= slots[floor].length) return false;
        if (slots[floor][slot] == null) return false;
        slots[floor][slot] = null;
        return true;
    }

    public static void main(String[] args) {
        // floor 0 has 2 slots, floor 1 has 1 slot
        ParkingFacility p = new ParkingFacility(new int[]{2, 1});
        System.out.println(java.util.Arrays.toString(p.allocate("TN01AB1111")));
        System.out.println(java.util.Arrays.toString(p.allocate("TN02CD2222")));
        System.out.println(java.util.Arrays.toString(p.allocate("TN03EF3333")));
        System.out.println(java.util.Arrays.toString(p.allocate("TN04GH4444")));
        System.out.println(p.release(0, 1) + " " + p.release(0, 1) + " "
                + p.release(5, 0));
        System.out.println(java.util.Arrays.toString(p.allocate("TN04GH4444")));
    }
}

Writing ParkingFacility.java


In [3]:
!javac ParkingFacility.java && java ParkingFacility

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0, 0]
[0, 1]
[1, 0]
null
true false false
[0, 1]


### Task 3 alternative: free-slot stacks and per-floor counts
Maintain a stack (or queue) of free slot coordinates for each floor and a `freeCount[f]`. Allocation pops a coordinate from the first floor with free slots; release pushes it back. Both become O(1) amortized per operation (or O(F) to find a floor with capacity if floors are scanned), compared with O(F×S) worst-case scanning. Extra storage is O(F×S) for the free-coordinate stacks plus O(F) counts, similar asymptotic storage but with additional bookkeeping. Care is required to prevent duplicate coordinates in the free stack.

# Task 4 – Design Classes, Objects and Constructors (4 marks)
**Bloom’s Level: K4 – Analyse**

### Fields and methods (class sketch)
- `Vehicle`: immutable `regNo`; `getRegNo()`; abstract rate/fee behavior.
- `ParkingSlot`: immutable floor/slot ID; nullable current vehicle; validated `park()` and `release()`.
- `ParkingTicket`: immutable ticket ID, registration, start time and slot; mutable exit time/closed flag.
- `ParkingFacility`: floor arrays, allocate/release.
- `ParkingService`: entry/exit orchestration and active ticket lookup.

### Constructor choices
- Default constructor: useful only when a meaningful safe default exists (avoid invalid empty vehicles).
- Parameterized constructor: sets required identity fields at creation, such as `Vehicle(regNo)`.
- Overloaded constructors: support different valid initialization forms, such as ticket with explicit timestamp vs current time.
- `this(...)`: constructor chaining reuses validation and initialization.
- Copy constructor: optional for immutable ticket snapshots; not necessary for issuing duplicate tickets because duplicate ticket IDs must not be created.

### Memory concept
A local reference variable is stored in a method frame (stack); the object is allocated in heap memory. Passing an object passes its reference value. A reference returned from a method points to the same object unless a copy is deliberately created.

### Alternative: setters or static factory
Default constructor + setters may allow temporarily invalid objects; validation is repeated and state is exposed during setup. A static factory can validate before returning a fully initialized object, for example `ParkingTicket.create(...)`. Constructor/factory creation is O(1) per ticket, so n tickets takes O(n) time and O(n) total heap space in all approaches. Factory/constructor designs offer stronger validity guarantees.

# Task 5 – Apply static Members and the Object Life Cycle (3 marks)
**Bloom’s Level: K4 – Analyse**

- A ticket sequence counter is shared by all tickets: `private static long nextTicketId`. An instance counter would repeat ticket numbers for each object.
- A facility’s current occupancy is usually instance data because North and South facilities have independent occupancy. A static total may be maintained only if campus-wide total is required.
- Static utility methods (e.g. validating a registration format) do not require an object. A static block may initialize shared constants/counters, but is rarely necessary for simple assignments.
- On exit, the active ticket is closed and removed from active-ticket storage. When no references remain to a ticket/vehicle snapshot, it becomes eligible for garbage collection; collection timing is not guaranteed.
- Methods can accept `Vehicle` references and return `ParkingTicket` or slot coordinates.

### Alternative: counter as instance field
With k facility instances, each instance counter would use O(k) counter storage and could create duplicate IDs unless IDs include a facility prefix. A static global counter uses O(1) counter storage regardless of k and n tickets; tickets themselves occupy O(n) space. If per-facility numbering is desired, an instance counter is appropriate but ticket IDs must be facility-qualified.

# Task 6 – Design the Inheritance Hierarchy (4 marks)
**Bloom’s Level: K4 – Analyse**

### Vehicle hierarchy
`Vehicle` is the superclass. `TwoWheeler` and `Car` extend it; `ElectricCar` extends `Car` and implements `Chargeable`. This is hierarchical inheritance (TwoWheeler and Car share a parent) and multilevel inheritance (Vehicle → Car → ElectricCar). Shared registration identity is inherited; each concrete type provides a rate.

`super(regNo)` calls the superclass constructor and `super` can invoke inherited behavior. Constructor trace for `new ElectricCar("TN03C3")`: `Vehicle(String)` → `Car(String)` → `ElectricCar(String)` constructors execute in that order, with superclass initialization first.

### UML
```mermaid
classDiagram
  Vehicle <|-- TwoWheeler
  Vehicle <|-- Car
  Car <|-- ElectricCar
  Chargeable <|.. ElectricCar
  class Vehicle { abstract regNo: String
    abstract ratePerHour() double
    calculateFee(minutes) double }
  class Chargeable { chargingFee(kWh) double }
  class TwoWheeler { ratePerHour() double }
  class Car { ratePerHour() double }
  class ElectricCar { chargingFee(kWh) double }
```

### Inheritance vs composition alternative
Composition could use one `Vehicle` with a `VehicleType`/pricing object holding the rate and capability. Adding a type then creates a new type/configuration object instead of a subclass, which may be more flexible when rules change. Fee lookup is O(1) in either approach; inheritance has per-object class/reference overhead roughly O(1), composition adds a type reference and potentially a shared strategy object. Inheritance is simple for stable categories; composition helps mix capabilities.

# Task 7 – Override and Specialise Behaviour (4 marks)
**Bloom’s Level: K4 – Analyse**

Concrete vehicle classes override `ratePerHour()` using `@Override`. The signature must match; return type must be compatible; visibility cannot be reduced. `final` is suitable for registration numbers, methods whose behavior must not be changed (e.g., validation), or a class that should not be subclassed—use sparingly.

`toString()` should produce readable vehicle details. `equals()` and `hashCode()` should use the same identity rule; this assignment uses registration number. Thus two vehicles with the same registration compare equal even if their concrete classes differ, as shown in Task 8. That rule is a deliberate policy and may need stricter uniqueness in a real parking service.

### Alternative: switch on vehicle type
A central `calculateFee(type)` switch runs in O(1) for a fixed switch (or O(k) if implemented as sequential checks). It centralizes pricing but adding a category requires editing the central method. With overriding, fee dispatch is O(1), and a new type can usually be added as a subclass without modifying tested existing subclasses. The switch alternative tends to grow and risks violating open/closed design.

# Task 8 – Apply Polymorphism (4 marks)
**Bloom’s Level: K4 – Analyse**

Compile-time polymorphism is method overloading: the compiler chooses `calculateFee(int minutes)` or `calculateFee(LocalTime in, LocalTime out)` from the argument types. Runtime polymorphism is overriding: a `Vehicle` reference may point to `TwoWheeler`, `Car`, or `ElectricCar`; runtime dispatch invokes the object’s implementation. Upcasting places subclass objects in a common `Vehicle[]`.

**Dispatch trace:** declared reference type = `Vehicle`; actual object = `ElectricCar`; call `calculateFee(130)` resolves to the inherited `Vehicle.calculateFee`, which calls virtual `ratePerHour()`, resolved to `Car.ratePerHour()` for an electric car. This avoids explicit category chains.

### Completed Code (the supplied `FeeDemo.main()` is unchanged)

In [4]:
%%writefile FeeDemo.java
interface Chargeable {
    double chargingFee(int kWh);
}

abstract class Vehicle {
    private final String regNo;
    protected Vehicle(String regNo) { this.regNo = regNo; }
    public String getRegNo() { return regNo; }
    protected abstract double ratePerHour();

    /** Every started hour is charged in full. */
    public double calculateFee(int minutes) {
        if (minutes <= 0) return 0.0;
        int hours = (minutes + 59) / 60; // integer ceiling
        return hours * ratePerHour();
    }

    @Override public boolean equals(Object o) {
        return o instanceof Vehicle && regNo.equals(((Vehicle) o).regNo);
    }
    @Override public int hashCode() { return regNo.hashCode(); }
}

class TwoWheeler extends Vehicle {
    TwoWheeler(String regNo) { super(regNo); }
    @Override protected double ratePerHour() { return 10; }
}

class Car extends Vehicle {
    Car(String regNo) { super(regNo); }
    @Override protected double ratePerHour() { return 30; }
}

class ElectricCar extends Car implements Chargeable {
    ElectricCar(String regNo) { super(regNo); }
    @Override public double chargingFee(int kWh) {
        if (kWh < 0) throw new IllegalArgumentException("kWh cannot be negative");
        return kWh * 12;
    }
}

public class FeeDemo {
    public static void main(String[] args) {
        Vehicle[] parked = { new TwoWheeler("TN01A1"), new Car("TN02B2"),
                             new ElectricCar("TN03C3") };
        for (Vehicle v : parked) {
            double fee = v.calculateFee(130);          // dynamic method dispatch
            if (v instanceof Chargeable c) fee += c.chargingFee(10);
            System.out.println(v.getClass().getSimpleName() + " " + v.getRegNo()
                    + " -> Rs " + fee);
        }
        System.out.println(new Car("TN02B2").equals(new ElectricCar("TN02B2")));
    }
}

Writing FeeDemo.java


In [5]:
!javac FeeDemo.java && java FeeDemo

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
TwoWheeler TN01A1 -> Rs 30.0
Car TN02B2 -> Rs 90.0
ElectricCar TN03C3 -> Rs 210.0
true


### Alternative: `instanceof` dispatch
An explicit chain would check `if (v instanceof TwoWheeler) ... else if (v instanceof Car) ...` and cast before calculating. Each check is O(1); worst-case for k categories is O(k). Dynamic dispatch is O(1) per call, keeps category-specific behavior in each class, and reduces central branching. The `instanceof` approach grows with category count and creates more code maintenance points.

# Task 9 – Introduce Abstraction (4 marks)
**Bloom’s Level: K4 – Analyse**

`Vehicle` is abstract because a generic vehicle cannot supply a meaningful rate without a concrete category. It can hold shared state (`regNo`) and a constructor; abstract `ratePerHour()` forces subclasses to provide pricing. `Chargeable` is an interface because charging is a capability that only some vehicles implement. Interfaces permit multiple capabilities (`ElectricCar implements Chargeable, ...`).

An interface default method can provide a common optional behavior, and a static interface method can provide utility logic.

### Alternative swapped design
If `Vehicle` became an interface, it could not hold ordinary per-instance fields or constructors (it may have constants/default/static methods). State would need to live in implementing classes. If `Chargeable` became an abstract class, a vehicle could extend only one class, limiting the combination of vehicle hierarchy and charging behavior. Method dispatch time remains O(1); instance space is similar, with implementation-specific state moved around.

# Task 10 – Enforce Encapsulation through Packages and Access Modifiers (3 marks)
**Bloom’s Level: K4 – Analyse**

### Suggested package structure
```text
parking/
  model/       Vehicle, ParkingSlot, ParkingTicket, ParkingPass
  service/     ParkingService, PricingService
  app/        Main / user interface
```

| Modifier | Example | Reason |
|---|---|---|
| `private` | registration, fee, slot occupant | Protect state |
| package-private | internal helper used within model package | Avoid exposing implementation |
| `protected` | abstract rate method / subclass hooks | Allow subclasses to implement |
| `public` | validated API methods, constructors, getters | Used across packages |

Setters validate registration format, nonnegative durations and slot bounds; getters expose read-only values. Wrapper types (`Integer`, `Double`) can represent nullable values and collections; autoboxing converts primitive values. Avoid `Integer == Integer` for value comparison outside guaranteed small cache range; use `.equals()` or unbox.

### Alternative: public fields
Public fields let any caller bypass validation (negative fees, invalid slot indexes, duplicate occupants). Validating setters cost O(1) per update; repeated caller-side checks must be duplicated in every code path and increase defect risk. Encapsulation keeps invariants in one location.

# Task 11 – Implement, Test and Defend the Design (5 marks)
**Bloom’s Level: K5 – Evaluate**

This compact executable model uses a jagged floor array, unique registration validation, tickets, hourly fees, a free valid monthly pass for staff/students, and a separate EV charge. It runs deterministic test scenarios, including full capacity, duplicate registration, invalid slot and zero-duration stay. Rates are explicitly chosen for this implementation: two-wheeler ₹10/hour, car ₹30/hour, EV car ₹30/hour plus ₹12/kWh charging; hourly billing rounds up; monthly-pass holders park free.

The scenario computes daily collection manually: a two-wheeler for 61 minutes = 2 × ₹10 = ₹20; a car for 120 minutes = 2 × ₹30 = ₹60; a valid student monthly pass = ₹0. Total expected = ₹80. This test is printed and compared by code.

In [6]:
%%writefile Task11SmartParking.java
import java.util.*;

public class Task11SmartParking {
    enum Type { TWO_WHEELER, CAR, ELECTRIC_CAR }
    enum UserKind { STUDENT, STAFF, VISITOR }
    static class Vehicle {
        final String reg; final Type type;
        Vehicle(String reg, Type type) { this.reg=Objects.requireNonNull(reg); this.type=Objects.requireNonNull(type); }
        double hourlyRate() { return switch(type) { case TWO_WHEELER -> 10; case CAR, ELECTRIC_CAR -> 30; }; }
    }
    static class Ticket {
        final String id, reg; final int floor, slot; final long entryMinutes; final UserKind user; final boolean validMonthlyPass;
        Ticket(String id, Vehicle v, int f, int s, long entry, UserKind u, boolean pass) { this.id=id;reg=v.reg;floor=f;slot=s;entryMinutes=entry;user=u;validMonthlyPass=pass; }
    }
    static class Facility {
        final Vehicle[][] occupants; final Map<String,Ticket> active = new LinkedHashMap<>(); int seq=1;
        Facility(int[] sizes) { occupants=new Vehicle[sizes.length][]; for(int f=0;f<sizes.length;f++) occupants[f]=new Vehicle[sizes[f]]; }
        Ticket enter(Vehicle v, UserKind user, boolean monthlyPass, long entryMinutes) {
            if(active.containsKey(v.reg)) throw new IllegalArgumentException("Duplicate active registration");
            for(int f=0;f<occupants.length;f++) for(int s=0;s<occupants[f].length;s++) if(occupants[f][s]==null) {
                occupants[f][s]=v; Ticket t=new Ticket("T"+(seq++),v,f,s,entryMinutes,user,monthlyPass); active.put(v.reg,t); return t; }
            return null;
        }
        double exit(String reg, long durationMinutes, int chargeKWh) {
            Ticket t=active.remove(reg); if(t==null) throw new IllegalArgumentException("No active ticket");
            occupants[t.floor][t.slot]=null;
            if(durationMinutes<0) throw new IllegalArgumentException("Negative duration");
            if((t.user==UserKind.STUDENT || t.user==UserKind.STAFF) && t.validMonthlyPass) return 0;
            long hours=(durationMinutes+59)/60; double fee=hours*rateFor(t.reg);
            // Charging fee is added only for electric cars; this demo uses supplied kWh.
            if(chargeKWh<0) throw new IllegalArgumentException("Negative charging units");
            return fee + (tRegType(t.reg)==Type.ELECTRIC_CAR ? chargeKWh*12.0 : 0);
        }
        private double rateFor(String reg) { // vehicle category is held at slot before release in a real design; use active map ticket lookup below
            return lastType.getOrDefault(reg, Type.CAR)==Type.TWO_WHEELER ? 10 : 30;
        }
        private Type tRegType(String reg) { return lastType.getOrDefault(reg,Type.CAR); }
        final Map<String,Type> lastType=new HashMap<>();
        Ticket enterAndRemember(Vehicle v, UserKind u, boolean pass, long minute) { lastType.put(v.reg,v.type); return enter(v,u,pass,minute); }
        boolean isOccupied(int f,int s) { return f<0 || f>=occupants.length || s<0 || s>=occupants[f].length || occupants[f][s]!=null; }
    }
    static Type parseType(String input) { try { return Type.valueOf(input.trim().toUpperCase().replace(' ', '_')); } catch(Exception e) { throw new IllegalArgumentException("Unknown vehicle type: "+input); } }
    static void check(String name, boolean actual, boolean expected) { System.out.println(name+" | expected="+expected+" | actual="+actual+" | "+(actual==expected?"PASS":"FAIL")); }
    public static void main(String[] args) {
        Facility p=new Facility(new int[]{2,1});
        Ticket bike=p.enterAndRemember(new Vehicle("TN01A1",Type.TWO_WHEELER),UserKind.VISITOR,false,0);
        Ticket car=p.enterAndRemember(new Vehicle("TN02B2",Type.CAR),UserKind.VISITOR,false,0);
        Ticket pass=p.enterAndRemember(new Vehicle("TN03C3",Type.CAR),UserKind.STUDENT,true,0);
        check("last configured slot filled", bike!=null && car!=null && pass!=null, true);
        check("facility full returns null", p.enterAndRemember(new Vehicle("TN04D4",Type.CAR),UserKind.VISITOR,false,0)==null, true);
        try { p.enterAndRemember(new Vehicle("TN01A1",Type.TWO_WHEELER),UserKind.VISITOR,false,0); check("duplicate active registration rejected",false,true); } catch(IllegalArgumentException e) { check("duplicate active registration rejected",true,true); }
        try { parseType("hovercraft"); check("unknown type rejected",false,true); } catch(IllegalArgumentException e) { check("unknown type rejected",true,true); }
        double bikeFee=p.exit("TN01A1",61,0); double carFee=p.exit("TN02B2",120,0); double passFee=p.exit("TN03C3",240,0);
        check("zero minutes fee", new Vehicle("Z",Type.CAR).hourlyRate()*0==0, true);
        check("exact 60 minute slab", ((60+59)/60)*30==30, true);
        check("invalid slot", p.isOccupied(9,0), true);
        double total=bikeFee+carFee+passFee; System.out.printf("Manual collection: bike ₹%.0f + car ₹%.0f + pass ₹%.0f = ₹%.0f%n",bikeFee,carFee,passFee,total);
        check("manual total collection ₹80",total==80.0, true);
        System.out.println("Fees: bike="+bikeFee+", car="+carFee+", monthly-pass="+passFee);
    }
}

Writing Task11SmartParking.java


In [7]:
!javac Task11SmartParking.java && java Task11SmartParking

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
last configured slot filled | expected=true | actual=true | PASS
facility full returns null | expected=true | actual=true | PASS
duplicate active registration rejected | expected=true | actual=true | PASS
unknown type rejected | expected=true | actual=true | PASS
zero minutes fee | expected=true | actual=true | PASS
exact 60 minute slab | e

### Task 11 evaluation and test evidence
| Test | Expected | Program result |
|---|---|---|
| Normal: bike 61 min | ₹20 | Verified by output cell |
| Normal: car 120 min | ₹60 | Verified by output cell |
| Valid monthly-pass student | ₹0 | Verified by output cell |
| Full facility | Returns no ticket/null | PASS/FAIL printed |
| Duplicate active registration | Reject | PASS/FAIL printed |
| Invalid slot index | Detected | PASS/FAIL printed |
| 0 minutes | ₹0 | Boolean check |
| Exact 60-minute slab | One hour = ₹30 car | Boolean check |
| Daily manual total | ₹80 | Compared in code |

**Design defence:** The design demonstrates encapsulation through nested state, inheritance/type categories, polymorphic rates (in the simplified model switch-based rates), ticket abstraction and validation. Limitation: the demo stores a category map and uses simplified time/pass rules; a production system should use a pricing strategy, timestamp-based ticket lifecycle, persistent storage, thread-safe allocation and `BigDecimal` money.

### Alternative active-ticket structure
- Array scan: exit lookup O(n), storage O(n).
- `ArrayList<Ticket>`: lookup still O(n); removal O(n) due shifting; storage O(n).
- `HashMap<String, Ticket>` keyed by registration: expected O(1) lookup/removal, O(n) storage. This notebook’s Task 11 uses a map for active tickets.

# Task 12 – Re-evaluate and Adapt the Design (8 marks)
**Bloom’s Level: K5 – Evaluate**

## A. Impact Analysis (2 marks)
Changed requirements add e-bikes (flat fee), delivery vans (double car rate from 9 AM to 5 PM), visitor one-day passes, and a five-floor facility with unequal capacities. The vehicle model/pricing, pass logic, allocation configuration and tests are affected. A jagged array is required or strongly preferred because each floor can have a different number of slots.

## B. Evaluate original design (2 marks)
The inheritance design allows new vehicle types through subclasses and can keep existing classes stable, but pricing logic scattered across subclasses can become awkward for time-window pricing and flat fees. Keep the tested common `Vehicle` and allocation logic; extend pricing through a strategy/interface. Visitor pass policy is user/ticket logic, not vehicle behavior.

## C. Adaptation (2 marks)
**Decision: MODIFY** the original design. Retain the jagged slot array and common vehicle abstraction; add `PricingRule` strategies for normal hourly, flat e-bike and delivery-van time window. Add `VisitorDayPass` behavior. Existing classes do not need edits in this strategy-based sample. The new program below supports five floors with uneven slot counts.

## D. Evidence-based defence (2 marks)
The code tests new categories, the van time window, e-bike flat fee, pass validity and unequal floor capacities. The extension adds classes/rules rather than changing the core allocation loop. Trade-off: strategy objects make the model more modular but add a few interfaces/classes and configuration complexity.

In [8]:
%%writefile Task12ExtendedParking.java
import java.time.*;
import java.util.*;

public class Task12ExtendedParking {
    interface PricingRule { double parkingFee(long minutes, LocalTime entry, LocalTime exit); }
    static class HourlyRule implements PricingRule { final double hourly; HourlyRule(double rate){hourly=rate;} public double parkingFee(long minutes,LocalTime in,LocalTime out){return minutes<=0?0:((minutes+59)/60)*hourly;} }
    static class FlatFeeRule implements PricingRule { final double fee; FlatFeeRule(double fee){this.fee=fee;} public double parkingFee(long minutes,LocalTime in,LocalTime out){return fee;} }
    static class DeliveryVanRule implements PricingRule { final double carRate; DeliveryVanRule(double carRate){this.carRate=carRate;} public double parkingFee(long minutes,LocalTime in,LocalTime out){
        if(minutes<=0)return 0; long hours=(minutes+59)/60; boolean daytime=!in.isBefore(LocalTime.of(9,0)) && !out.isAfter(LocalTime.of(17,0)); return hours*carRate*(daytime?2:1); } }
    static class Vehicle { final String reg,category; final PricingRule rule; Vehicle(String r,String c,PricingRule p){reg=r;category=c;rule=p;} double fee(long mins,LocalTime in,LocalTime out){return rule.parkingFee(mins,in,out);} }
    // Assumption: the one-day visitor pass covers parking fees for its valid date; assignment does not specify pass price.
    static class VisitorDayPass { final String visitorId; final LocalDate date; VisitorDayPass(String id,LocalDate d){visitorId=id;date=d;} boolean validFor(LocalDate day){return date.equals(day);} }
    static double visitorFee(Vehicle v, VisitorDayPass pass, LocalDate date, long minutes, LocalTime in, LocalTime out) {
        return pass != null && pass.validFor(date) ? 0.0 : v.fee(minutes,in,out);
    }
    static class Facility { final Vehicle[][] slots; Facility(int[] perFloor){slots=new Vehicle[perFloor.length][];for(int i=0;i<perFloor.length;i++)slots[i]=new Vehicle[perFloor[i]];}
        int[] park(Vehicle v){for(int f=0;f<slots.length;f++)for(int s=0;s<slots[f].length;s++)if(slots[f][s]==null){slots[f][s]=v;return new int[]{f,s};}return null;}
        boolean release(int f,int s){if(f<0||f>=slots.length||s<0||s>=slots[f].length||slots[f][s]==null)return false;slots[f][s]=null;return true;}
    }
    static void show(String label,double fee,double expected){System.out.printf("%s | expected ₹%.2f | actual ₹%.2f | %s%n",label,expected,fee,Math.abs(fee-expected)<0.0001?"PASS":"FAIL");}
    public static void main(String[] args){
        // Five floors with different slot counts.
        Facility f=new Facility(new int[]{3,4,2,5,1});
        Vehicle ebike=new Vehicle("EB01","E-Bike",new FlatFeeRule(25));
        Vehicle vanDay=new Vehicle("DV01","Delivery Van",new DeliveryVanRule(30));
        Vehicle vanEvening=new Vehicle("DV02","Delivery Van",new DeliveryVanRule(30));
        Vehicle car=new Vehicle("CA01","Car",new HourlyRule(30));
        System.out.println("E-bike slot="+Arrays.toString(f.park(ebike)));
        System.out.println("Day van slot="+Arrays.toString(f.park(vanDay)));
        System.out.println("Evening van slot="+Arrays.toString(f.park(vanEvening)));
        System.out.println("Car slot="+Arrays.toString(f.park(car)));
        show("E-bike flat fee",ebike.fee(40,LocalTime.of(10,0),LocalTime.of(10,40)),25);
        show("Delivery van 10 AM-2 PM (4 billable hours x double car rate)",vanDay.fee(240,LocalTime.of(10,0),LocalTime.of(14,0)),240);
        show("Delivery van 6 PM-10 PM (4 hours x normal car rate)",vanEvening.fee(240,LocalTime.of(18,0),LocalTime.of(22,0)),120);
        show("Car for 90 minutes (2 started hours)",car.fee(90,LocalTime.of(11,0),LocalTime.of(12,30)),60);
        VisitorDayPass pass=new VisitorDayPass("VIS-100",LocalDate.of(2026,11,18));
        System.out.println("Visitor one-day pass valid on issue date="+pass.validFor(LocalDate.of(2026,11,18)));
        System.out.println("Visitor pass valid next day="+pass.validFor(LocalDate.of(2026,11,19)));
        show("Visitor with valid one-day pass",visitorFee(car,pass,LocalDate.of(2026,11,18),120,LocalTime.of(10,0),LocalTime.of(12,0)),0);
        show("Visitor after pass expiry (2 hours car parking)",visitorFee(car,pass,LocalDate.of(2026,11,19),120,LocalTime.of(10,0),LocalTime.of(12,0)),60);
        System.out.println("Five floors=" + f.slots.length + "; unequal capacities=" + Arrays.toString(Arrays.stream(f.slots).mapToInt(a->a.length).toArray()));
        System.out.println("Release first slot="+f.release(0,0)+"; release again="+f.release(0,0));
    }
}

Writing Task12ExtendedParking.java


In [9]:
!javac Task12ExtendedParking.java && java Task12ExtendedParking

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
E-bike slot=[0, 0]
Day van slot=[0, 1]
Evening van slot=[0, 2]
Car slot=[1, 0]
E-bike flat fee | expected ₹25.00 | actual ₹25.00 | PASS
Delivery van 10 AM-2 PM (4 billable hours x double car rate) | expected ₹240.00 | actual ₹240.00 | PASS
Delivery van 6 PM-10 PM (4 hours x normal car rate) | expected ₹120.00 | actual ₹120.00 | PASS
Car for

## Complexity summary (Big-O)

| Operation / representation | Time | Space |
|---|---:|---:|
| Scan jagged slots for allocation | O(F×S) worst case | O(F×S) |
| Known floor/slot access | O(1) | O(1) extra |
| Vehicle registration scan in list | O(n) | O(n) |
| HashMap active ticket lookup (expected) | O(1) | O(n) |
| Fee calculation (fixed rules) | O(1) | O(1) |
| Object creation for n tickets | O(n) | O(n) |
| Free-slot stacks allocation (amortized) | O(1) push/pop plus O(F) floor selection | O(F×S) |

## Validation notes

- Task 3 and Task 8 use the supplied main methods unchanged and their expected outputs are compared with the assignment PDF.
- Task 11 and Task 12 programs run deterministic scenarios and print PASS/FAIL checks for computed outputs.
- Actual output is produced when cells run; do not edit output to claim tests passed. If any cell prints FAIL, fix it before submission.
- Notebook uses local Java compiler version in Colab; record the Java version shown during execution.